In [1]:
import lxml.html
import requests
import time
import uuid
import pandas as pd

In [2]:
def make_request(url):
    headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
    }

    resp = requests.get(url, headers = headers)
    print(f"Scraped with status code {resp.status_code}")
    print(f"Response headers: {resp.headers}")

    if resp.status_code == 429:
        time.sleep(2.5)
        resp = make_request(url)
    elif resp.status_code == 200:
        return resp

In [3]:
def scrape_judge(url):
    j_page = lxml.html.fromstring(make_request(url).text)
    r_d = {}

    item_titles = j_page.xpath("//div[@class = 'judge-info']//div[@class = 'about-list']//div[@class = 'about-list-item']//h3/text()")
    items = j_page.xpath("//div[@class = 'judge-info']//div[@class = 'about-list']//div[@class = 'about-list-item']//p/text()")

    for i, title in enumerate(item_titles):
        r_d[title.lower()] = items[i].replace("\t", "").strip().lower()

    return r_d

In [4]:
url = "https://state-law-research.org/state-justices/"

root = lxml.html.fromstring(make_request(url).text)

judge_links = root.xpath("//section[@filter = 'judge']//a[contains(@href, 'judge')]//@href")
judge_names = root.xpath("//section[@filter = 'judge']//a[contains(@href, 'judge')]//div[@class = 'module--content module--content-post']//h2[@class = 'title']/text()")

judge_pd = {
    "JID": [],
    "name": []
}

Scraped with status code 200
Response headers: {'Date': 'Mon, 13 Apr 2026 22:12:57 GMT', 'Content-Type': 'text/html; charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Content-Encoding': 'gzip', 'Vary': 'Accept-Encoding, Accept-Encoding, Accept-Encoding, Accept-Encoding,Cookie', 'Link': '<https://state-law-research.org/wp-json/>; rel="https://api.w.org/", <https://state-law-research.org/wp-json/wp/v2/pages/2560>; rel="alternate"; title="JSON"; type="application/json", <https://state-law-research.org/?p=2560>; rel=shortlink', 'X-Powered-By': 'WP Engine', 'X-Cacheable': 'SHORT', 'Cache-Control': 'max-age=600, must-revalidate', 'X-Cache': 'HIT: 1', 'X-Cache-Group': 'normal', 'cf-cache-status': 'DYNAMIC', 'Set-Cookie': '__cf_bm=.k2AeU1C8J3aU4WWLKiM1685Hpe.dUyUWNaBLj.fid4-1776118377-1.0.1.1-LO7.gcq366lFM9E8SOj3FhG5bzpuPVCSB.QX2W1JP5XbMmGtDsfH33AgAhHKZunrp3pu2bnRFFeX_.RhpHMGW6Rii2t16yStEqfHG9BtW_Y; path=/; expires=Mon, 13-Apr-26 22:42:57 GMT; domain=.state-law-resea

In [5]:
for i, name in enumerate(judge_names):
    judge_pd["name"].append(name)
    judge_pd["JID"].append(uuid.uuid4())

    judge_dic = scrape_judge(judge_links[i])

    if judge_dic == {}:
        break

    for item in judge_dic.keys():
        data = judge_dic[item]
        if data is None:
            data
        if not (item in judge_pd):
            judge_pd[item] = [judge_dic[item]]
        else:
            judge_pd[item].append(judge_dic[item])

Scraped with status code 200
Response headers: {'Date': 'Mon, 13 Apr 2026 22:13:24 GMT', 'Content-Type': 'text/html; charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Content-Encoding': 'gzip', 'Vary': 'Accept-Encoding, Accept-Encoding, Accept-Encoding, Accept-Encoding,Cookie', 'Link': '<https://state-law-research.org/wp-json/>; rel="https://api.w.org/", <https://state-law-research.org/wp-json/wp/v2/judge/1944>; rel="alternate"; title="JSON"; type="application/json", <https://state-law-research.org/?p=1944>; rel=shortlink', 'X-Powered-By': 'WP Engine', 'X-Cacheable': 'SHORT', 'Cache-Control': 'max-age=600, must-revalidate', 'X-Cache': 'HIT: 1', 'X-Cache-Group': 'normal', 'cf-cache-status': 'DYNAMIC', 'Set-Cookie': '__cf_bm=UK68gi7xNB96KkB65L159YFIhNr2WkLtlpCMkrJZKwY-1776118404-1.0.1.1-BWdYy2DlM95T_7kl4Dd7igjng6Kaj1mLSxEA7MNeDTlvDuD80_xNf1dyvlPtCZwQ0oPx_PYOnGBcSdTTa.w1orPtMQYlIzRXqhksq584OfU; path=/; expires=Mon, 13-Apr-26 22:43:24 GMT; domain=.state-law-resea

In [6]:
for key in judge_pd.keys():
    print(f"{key}: {len(judge_pd[key])}")

JID: 345
name: 345
gender: 345
party: 343
race: 345
professional experience: 345
election type: 345
term start: 345
term end: 335
next election date: 323


In [7]:
del judge_pd["party"]
del judge_pd["next election date"]
del judge_pd["term end"]

In [8]:
pd.DataFrame(judge_pd)

,JID,name,gender,race,professional experience,election type,term start
0,cf522304-b4de-47e4-b6d0-1e70d34501c1,Abigail LeGrow,female,white,"corporate, judicial","appointed, leg confirmed","may 3, 2023"
1,8da3e8c4-59d3-4dcb-b178-0b4f345393dd,Adam Tanenbaum,male,white,"government civil (non civil rights), judicial,...",appointed,"january 14, 2026"
2,c7c49510-f094-4f5f-a197-24a162c73dd6,Aimee A. Oravec,female,white,"corporate, misc. private practice",appointed,"january 31, 2025"
3,b30ea508-88c6-4507-8f21-477edbcc55a9,Alisa Kelli Wise,female,white,"corporate, misc. private practice, judicial, o...","elected, partisan",2011
4,123d0562-0abd-4978-acb3-3f7fc61192c9,Allison Riggs,female,white,"legal aid/non-govt civil rights, judicial",appointed,"september 11, 2023"
...,...,...,...,...,...,...,...
340,e808ee8f-f9b3-4d01-84c0-7d5d5d0bca2a,"William J. Musseman, Jr.",male,white,"prosecutor, judicial",appointed,2022
341,6c50aa32-3890-4f1d-bca6-ea51c16c3d04,William P. Robinson III,male,white,misc. private practice,"appointed, leg confirmed",2004
342,a3c75d77-5658-4fdf-a135-a69b4ae831f8,William R. Wooton,male,white,"prosecutor, plaintiff-side civil private pract...","elected, nonpartisan","january 1, 2021"
343,28c7d97b-630f-4dc9-a7cf-2e048650f1c9,"William W. Hood, III",male,white,"prosecutor, corporate, plaintiff-side civil pr...","appointed, retention elected","january 13, 2014"
